# Lab 2 — Fine-tune Parakeet 0.6B with NVIDIA NeMo

This lab follows NVIDIA Riva's `asr-finetune-parakeet-nemo.ipynb` pattern: prepare NeMo manifests, reuse the pretrained tokenizer for a small dataset, fine-tune with NeMo, evaluate held-out speech, and save a complete `.nemo` model for Riva. We use Dutch FLEURS as a visible cross-language adaptation exercise. It is educational evidence, not a production Dutch accuracy claim.


In [ ]:
from pathlib import Path
import json, os, sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache' / 'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import torch
import nemo.collections.asr as nemo_asr
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint
from omegaconf import OmegaConf

from voice_asr_lab.audio import duration_seconds, load_dummy_librispeech, load_fleurs_records, total_duration
from voice_asr_lab.nemo import (
    configure_nemo_trainable_parameters, evaluate_nemo_manifest,
    nemo_tokenizer_coverage, write_nemo_manifest,
)
from voice_asr_lab.profiles import detect_profile

assert torch.cuda.is_available(), 'This NeMo lab requires an NVIDIA GPU.'


## 1. Choose the experiment controls

These are the workshop knobs. Increase examples and optimizer steps for a longer run. Keep validation and test as separate official splits, and do not tune controls after inspecting test results.


In [ ]:
profile = detect_profile()

MODEL_ID = 'nvidia/parakeet-ctc-0.6b'
LANGUAGE_CONFIG = 'nl_nl'
LANGUAGE_NAME = 'Dutch'
TRAIN_EXAMPLES = 80 if profile.name == 't4' else 200
VALIDATION_EXAMPLES = 20 if profile.name == 't4' else 30
TEST_EXAMPLES = 30 if profile.name == 't4' else 50
ENGLISH_GUARDRAIL_EXAMPLES = 4
MAX_STEPS = 150 if profile.name == 't4' else 200
VAL_CHECK_INTERVAL = 25
LEARNING_RATE = 1e-4 if profile.name == 't4' else 5e-5
TRAINABLE_ENCODER_LAYERS = 0 if profile.name == 't4' else 2  # -1 trains all layers
TRAIN_BATCH_SIZE = profile.train_batch_size
EVAL_BATCH_SIZE = 4 if profile.name == 't4' else 8
ACCUMULATE_GRAD_BATCHES = 4 if profile.name == 't4' else 2
RANDOM_SEED = 7

assert 0 < VAL_CHECK_INTERVAL <= MAX_STEPS
assert TRAINABLE_ENCODER_LAYERS >= -1
precision = 'bf16-mixed' if torch.cuda.is_bf16_supported() else '16-mixed'
seed_everything(RANDOM_SEED, workers=True)
print({
    'profile': profile.name, 'model': MODEL_ID, 'language': LANGUAGE_CONFIG,
    'examples': (TRAIN_EXAMPLES, VALIDATION_EXAMPLES, TEST_EXAMPLES),
    'max_optimizer_steps': MAX_STEPS, 'validation_every_train_batches': VAL_CHECK_INTERVAL,
    'learning_rate': LEARNING_RATE, 'precision': precision,
    'trainable_encoder_layers': TRAINABLE_ENCODER_LAYERS,
    'effective_batch_size': TRAIN_BATCH_SIZE * ACCUMULATE_GRAD_BATCHES,
})


## 2. Prepare NeMo manifests from official FLEURS splits

NeMo expects one JSON object per audio file with `audio_filepath`, `duration`, and `text`. Audio is written as lossless 16 kHz PCM WAV. Dutch text is normalized to the English checkpoint's lower-case Latin output contract.


In [ ]:
train_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'train', limit=TRAIN_EXAMPLES, max_audio_seconds=profile.max_audio_seconds
)
validation_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'validation', limit=VALIDATION_EXAMPLES, max_audio_seconds=profile.max_audio_seconds
)
test_records = load_fleurs_records(
    LANGUAGE_CONFIG, 'test', limit=TEST_EXAMPLES, max_audio_seconds=profile.max_audio_seconds
)
english_records = [
    row for row in load_dummy_librispeech(limit=30)
    if duration_seconds(row['audio'], row['sampling_rate']) <= profile.max_audio_seconds
][:ENGLISH_GUARDRAIL_EXAMPLES]
if len(english_records) != ENGLISH_GUARDRAIL_EXAMPLES:
    raise RuntimeError('Not enough duration-safe English guardrail examples.')

manifest_dir = ROOT / 'artifacts' / 'nemo_manifests'
train_manifest = write_nemo_manifest(train_records, manifest_dir, 'nl_train')
validation_manifest = write_nemo_manifest(validation_records, manifest_dir, 'nl_validation')
test_manifest = write_nemo_manifest(test_records, manifest_dir, 'nl_test')
english_manifest = write_nemo_manifest(english_records, manifest_dir, 'en_guardrail')
print({
    'train_minutes': round(total_duration(train_records) / 60, 1),
    'validation_minutes': round(total_duration(validation_records) / 60, 1),
    'test_minutes': round(total_duration(test_records) / 60, 1),
    'train_manifest': str(train_manifest),
})
print('Original:  ', train_records[0]['original_text'])
print('Normalized:', train_records[0]['text'])


## 3. Load the NeMo checkpoint and audit the reused tokenizer

The NVIDIA reference recommends reusing the pretrained tokenizer when adaptation data is below 50 hours. We intentionally skip tokenizer training, but stop if any normalized transcript produces an unknown token.


In [ ]:
model = nemo_asr.models.ASRModel.from_pretrained(MODEL_ID, map_location='cuda')
model = model.to('cuda')
coverage = nemo_tokenizer_coverage(
    model.tokenizer,
    [row['text'] for row in train_records + validation_records + test_records],
)
print({'model_class': type(model).__name__, 'tokenizer_coverage': coverage})
if coverage['unknown_tokens']:
    raise RuntimeError('Tokenizer coverage failed. Inspect affected_examples before training.')


## 4. Measure the untouched base model

Validation selects the checkpoint. Test is reserved for the final before/after comparison. The English slice is a small catastrophic-forgetting warning, not a full English benchmark.


In [ ]:
model.eval()
baseline_validation = evaluate_nemo_manifest(model, validation_manifest, EVAL_BATCH_SIZE)
baseline_test = evaluate_nemo_manifest(model, test_manifest, EVAL_BATCH_SIZE)
baseline_english = evaluate_nemo_manifest(model, english_manifest, EVAL_BATCH_SIZE)
print({
    'baseline_validation_wer': baseline_validation['wer'],
    'baseline_test_wer': baseline_test['wer'],
    'baseline_test_cer': baseline_test['cer'],
    'baseline_english_wer': baseline_english['wer'],
})
for reference, prediction in list(zip(
    baseline_test['references'], baseline_test['predictions']
))[:3]:
    print(f'REF: {reference}\nHYP: {prediction}\n')


## 5. Fine-tune with NeMo and select on `val_wer`

The CTC decoder and a configurable tail of encoder layers are trainable. Lightning saves the best local checkpoint by validation WER. Increase `MAX_STEPS`, sample counts, or trainable layers only when the GPU and workshop duration allow it.


In [ ]:
parameter_summary = configure_nemo_trainable_parameters(
    model, TRAINABLE_ENCODER_LAYERS
)
checkpoint_dir = ROOT / 'artifacts' / 'nemo_checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_callback = ModelCheckpoint(
    dirpath=checkpoint_dir,
    filename='parakeet-nl-{step:04d}-{val_wer:.4f}',
    monitor='val_wer', mode='min', save_top_k=1, save_last=False,
)
trainer = Trainer(
    accelerator='gpu', devices=1, precision=precision,
    max_steps=MAX_STEPS, max_epochs=-1,
    val_check_interval=VAL_CHECK_INTERVAL,
    accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
    gradient_clip_val=1.0, callbacks=[checkpoint_callback],
    logger=False, log_every_n_steps=10, num_sanity_val_steps=0,
    enable_progress_bar=True, deterministic='warn',
)
model.set_trainer(trainer)
model.setup_training_data(train_data_config={
    'manifest_filepath': str(train_manifest), 'sample_rate': 16000,
    'batch_size': TRAIN_BATCH_SIZE, 'shuffle': True,
    'num_workers': 2, 'pin_memory': True,
})
model.setup_validation_data(val_data_config={
    'manifest_filepath': str(validation_manifest), 'sample_rate': 16000,
    'batch_size': EVAL_BATCH_SIZE, 'shuffle': False,
    'num_workers': 2, 'pin_memory': True,
})
model.setup_optimization(optim_config=OmegaConf.create({
    'name': 'adamw', 'lr': LEARNING_RATE, 'betas': [0.9, 0.98],
    'weight_decay': 0.001,
    'sched': {
        'name': 'CosineAnnealing', 'warmup_steps': min(25, MAX_STEPS // 10),
        'min_lr': LEARNING_RATE / 20, 'max_steps': MAX_STEPS,
    },
}))
print(parameter_summary)
trainer.fit(model)


## 6. Restore the best checkpoint and save a complete `.nemo` model

The `.nemo` file contains the model configuration, pretrained tokenizer, and selected weights. Lab 3 converts this artifact into a deployable Riva RMIR.


In [ ]:
best_checkpoint = Path(checkpoint_callback.best_model_path)
if not best_checkpoint.is_file():
    raise RuntimeError('No best checkpoint was saved; inspect val_wer in the training log.')
trusted_checkpoint = torch.load(best_checkpoint, map_location='cpu', weights_only=False)
model.load_state_dict(trusted_checkpoint['state_dict'], strict=True)
model = model.to('cuda').eval()
nemo_artifact = ROOT / 'artifacts' / 'parakeet-ctc-0.6b-nl.nemo'
model.save_to(str(nemo_artifact))
print({
    'best_checkpoint': str(best_checkpoint),
    'best_validation_wer': float(checkpoint_callback.best_model_score),
    'nemo_artifact': str(nemo_artifact),
    'size_gb': round(nemo_artifact.stat().st_size / 1024**3, 2),
})


## 7. Final held-out comparison and English guardrail


In [ ]:
selected_validation = evaluate_nemo_manifest(model, validation_manifest, EVAL_BATCH_SIZE)
selected_test = evaluate_nemo_manifest(model, test_manifest, EVAL_BATCH_SIZE)
selected_english = evaluate_nemo_manifest(model, english_manifest, EVAL_BATCH_SIZE)
summary = {
    'model_id': MODEL_ID, 'language': LANGUAGE_NAME, 'language_config': LANGUAGE_CONFIG,
    'baseline_validation_wer': baseline_validation['wer'],
    'selected_validation_wer': selected_validation['wer'],
    'baseline_test_wer': baseline_test['wer'],
    'selected_test_wer': selected_test['wer'],
    'test_wer_absolute_improvement': baseline_test['wer'] - selected_test['wer'],
    'test_wer_relative_improvement': (
        (baseline_test['wer'] - selected_test['wer']) / baseline_test['wer']
        if baseline_test['wer'] else 0.0
    ),
    'baseline_english_wer': baseline_english['wer'],
    'selected_english_wer': selected_english['wer'],
    'nemo_artifact': str(nemo_artifact),
    'tokenizer_retrained': False, 'tokenizer_coverage': coverage,
    'max_steps': MAX_STEPS, 'train_examples': TRAIN_EXAMPLES,
    'validation_examples': VALIDATION_EXAMPLES, 'test_examples': TEST_EXAMPLES,
    'trainable_encoder_layers': TRAINABLE_ENCODER_LAYERS,
}
summary_path = ROOT / 'artifacts' / 'lab2_run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
summary


In [ ]:
changed = 0
for reference, before, after in zip(
    baseline_test['references'], baseline_test['predictions'], selected_test['predictions']
):
    if before != after:
        print(f'REF:    {reference}\nBEFORE: {before}\nAFTER:  {after}\n')
        changed += 1
    if changed == 5:
        break
if changed == 0:
    print('No held-out transcript changed. Report this honestly and adjust only from validation evidence.')


## Interpretation and handoff

A positive held-out WER improvement plus an acceptable English guardrail is evidence for this bounded experiment. The same-tokenizer path deliberately avoids vocabulary surgery and keeps the decoder shape compatible with the pretrained checkpoint. It does not turn a small Dutch subset into a production multilingual model. Lab 3 uses NVIDIA Riva ServiceMaker to build an RMIR and deploys the Riva API, which is backed internally by TensorRT and Triton.
